# 07 - Proyek Akhir Kursus Coursera: Klasifikasi

> Ini proyek akhir **kursus Coursera** (Modul 6, "Gambaran Umum Tugas Akhir"), bukan topik TA mata kuliah.

Instruksi umum: pilih dataset klasifikasi, lakukan **prapemrosesan**, bangun dan **bandingkan beberapa model klasifikasi**, evaluasi dengan metrik yang tepat, lalu tulis ringkasan proses dan kesimpulan.

## 1. Latar Belakang & Rumusan Masalah
(...)

## 2. Dataset
(sumber, jumlah data, fitur, target, apakah kelas seimbang) - contoh di bawah memakai Breast Cancer; ganti dengan dataset pilihanmu.

## 3. Metodologi
(Prapemrosesan -> Split bertingkat -> Modeling (Regresi Logistik, KNN, SVM, Pohon, Random Forest, Gradient Boosting, Stacking) -> Validasi silang -> Tuning -> Evaluasi -> Kesimpulan)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_curve, roc_auc_score, precision_recall_curve, average_precision_score)

sns.set_theme(style='whitegrid')

# Ganti path dan nama target kalau pakai dataset lain
df = pd.read_csv('../datasets/breast_cancer.csv')
TARGET = 'target'
df.head()

## 4. Eksplorasi & Prapemrosesan

In [ ]:
print(df.shape)
print('Missing value:', df.isna().sum().sum())
print('Data duplikat:', df.duplicated().sum())
print()
print(df[TARGET].value_counts())
print('Proporsi kelas positif:', round(df[TARGET].mean(), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x=TARGET, data=df, ax=ax[0]); ax[0].set_title('Distribusi kelas')
korelasi_target = df.corr()[TARGET].drop(TARGET).sort_values()
pd.concat([korelasi_target.head(7), korelasi_target.tail(7)]).plot(kind='barh', ax=ax[1])
ax[1].set_title('Korelasi fitur dengan target (14 teratas)')
plt.tight_layout(); plt.show()

In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## 5. Modeling: bandingkan banyak model dengan validasi silang bertingkat

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dasar = [
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ('svm', make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
]

kandidat = {
    'Regresi Logistik': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    'SVM (RBF)': make_pipeline(StandardScaler(), SVC(random_state=42)),
    'Pohon Keputusan': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Stacking': StackingClassifier(estimators=dasar, final_estimator=LogisticRegression(max_iter=2000), cv=5),
}

hasil = []
for nama, m in kandidat.items():
    h = cross_validate(m, X_train, y_train, cv=kf, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
    hasil.append([nama, h['test_accuracy'].mean(), h['test_precision'].mean(), h['test_recall'].mean(), h['test_f1'].mean(), h['test_roc_auc'].mean()])

tabel = pd.DataFrame(hasil, columns=['Model', 'Akurasi', 'Presisi', 'Recall', 'F1', 'ROC-AUC']).set_index('Model').round(4)
tabel.sort_values('F1', ascending=False)

### Tuning model terbaik (GridSearchCV)
Contoh: SVM RBF. Ganti dengan model yang paling bagus di tabel.

In [ ]:
pipe = make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
param = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.001, 0.01, 0.1]}

grid = GridSearchCV(pipe, param, cv=kf, scoring='f1')
grid.fit(X_train, y_train)
print('Parameter terbaik:', grid.best_params_)
print('F1 CV            :', round(grid.best_score_, 4))

## 6. Evaluasi pada data test

In [ ]:
final = grid.best_estimator_
pred = final.predict(X_test)
proba = final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, target_names=['jinak', 'ganas']))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
print('PR-AUC :', round(average_precision_score(y_test, proba), 4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['jinak', 'ganas']).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Confusion matrix')

fpr, tpr, _ = roc_curve(y_test, proba)
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], 'r--')
ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR'); ax[1].set_title('Kurva ROC')

prec, rec, _ = precision_recall_curve(y_test, proba)
ax[2].plot(rec, prec); ax[2].set_xlabel('Recall'); ax[2].set_ylabel('Presisi'); ax[2].set_title('Kurva Presisi-Recall')
plt.tight_layout(); plt.show()

## 7. Insight: fitur apa yang paling berpengaruh?
Permutation importance (model-agnostik) bisa dipakai untuk model apa pun.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final, X_test, y_test, n_repeats=10, random_state=42, scoring='f1')
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()
imp.tail(10).plot(kind='barh', figsize=(7, 4))
plt.title('10 fitur terpenting (permutation importance)'); plt.show()

## 8. Kesimpulan & Langkah Selanjutnya
- Model terbaik dan alasannya (pilih metrik yang sesuai kasus): (...)
- Temuan utama / fitur berpengaruh: (...)
- Keterbatasan: (...)
- Langkah selanjutnya: (misal tangani kelas tidak seimbang, tambah data, coba model lain, threshold tuning, dll.)